# FontTools Package Examples

FontTools is a library for manipulating fonts, written in Python. It supports TTF, OTF, and other font formats, and provides tools for font subsetting, merging, and feature development.

In [ ]:
# Example 1: Basic Font Loading and Inspection
import fontTools
from fontTools.ttLib import TTFont
from fontTools.fontBuilder import FontBuilder
import tempfile
import os

print(f"FontTools version: {fontTools.__version__}")

# Build a small TrueType font, write it to disk and load it back with TTFont (offline).
fb = FontBuilder(1000, isTTF=True)
fb.setupGlyphOrder([".notdef", "A", "B", "space"])
fb.setupCharacterMap({ord('A'): 'A', ord('B'): 'B', ord(' '): 'space'})
fb.setupNameTable({"familyName": "TestFont", "styleName": "Regular"})
fb.setupHorizontalMetrics({".notdef": (500, 50), "A": (600, 50), "B": (620, 50), "space": (300, 0)})
fb.setupHorizontalHeader()
fb.setupOS2()
fb.setupPost()

with tempfile.TemporaryDirectory() as tmp_dir:
    font_path = os.path.join(tmp_dir, "TestFont-Regular.ttf")
    fb.save(font_path)
    
    # Load font with TTFont
    font = TTFont(font_path)
    
    # Basic font information
    print(f"Number of glyphs: {len(font.getGlyphNames())}")
    print(f"First 10 glyph names: {font.getGlyphNames()[:10]}")
    
    # Font tables information
    print(f"Available tables: {sorted(font.keys())}")
    
    # Get font metrics from 'head' table
    head_table = font['head']
    print(f"Font revision: {head_table.fontRevision}")
    print(f"Units per EM: {head_table.unitsPerEm}")
    
    # Get name table information
    name_table = font['name']
    font_name = name_table.getDebugName(1)  # Font family name
    print(f"Font family: {font_name}")
    print(f"Character map: {dict(font.getBestCmap())}")
    
    assert font.getGlyphOrder() == [".notdef", "A", "B", "space"]
    assert head_table.unitsPerEm == 1000 and font_name == "TestFont"
    assert dict(font.getBestCmap()) == {32: 'space', 65: 'A', 66: 'B'}
    font.close()

print("✅ Font loading and inspection completed")

## Example 2: Font Subsetting for Web Optimization

In [ ]:
# Example 2: Font Subsetting for Web Optimization
from fontTools.subset import Subsetter, Options
from fontTools.ttLib import TTFont
from fontTools.fontBuilder import FontBuilder
import tempfile
import os

# Create a sample font for subsetting
fb = FontBuilder(1000, isTTF=True)

# Setup glyph order
glyphs_to_add = [".notdef", "A", "B", "C", "D", "E", "space", "period", "comma"]
fb.setupGlyphOrder(glyphs_to_add)

# Setup character map
char_map = {
    ord('A'): 'A', ord('B'): 'B', ord('C'): 'C', 
    ord('D'): 'D', ord('E'): 'E', ord(' '): 'space',
    ord('.'): 'period', ord(','): 'comma'
}
fb.setupCharacterMap(char_map)

# Set font metadata
fb.setupNameTable({"familyName": "SubsetTest", "styleName": "Regular"})

# Setup metrics
metrics = {glyph: (600, 50) for glyph in glyphs_to_add}
fb.setupHorizontalMetrics(metrics)

# Setup required tables
fb.setupHorizontalHeader()
fb.setupOS2()
fb.setupPost()

# Create the font
original_font = fb.font

print(f"Original font glyphs: {len(original_font.getGlyphNames())}")
print(f"Original glyph names: {original_font.getGlyphNames()}")

# Save font to file and load for subsetting
with tempfile.NamedTemporaryFile(suffix='.ttf', delete=False) as temp_file:
    try:
        original_font.save(temp_file.name)
        
        # Load font for subsetting
        subset_font = TTFont(temp_file.name)
        
        # Create subsetter and subset to "ABC"
        subsetter = Subsetter()
        subsetter.populate(text="ABC")
        subsetter.subset(subset_font)
        
        print(f"\nSubset font glyphs: {len(subset_font.getGlyphNames())}")
        print(f"Subset glyph names: {subset_font.getGlyphNames()}")
        
        # Demonstrate different subsetting approaches
        print("\n--- Different Subsetting Methods ---")
        
        # Method 1: Unicode codepoints
        subset_font2 = TTFont(temp_file.name)
        subsetter2 = Subsetter()
        subsetter2.populate(unicodes=[ord('A'), ord('B')])
        subsetter2.subset(subset_font2)
        print(f"Unicode subset (A,B): {subset_font2.getGlyphNames()}")
        
        # Method 2: Glyph names
        subset_font3 = TTFont(temp_file.name)
        subsetter3 = Subsetter()
        subsetter3.populate(glyphs=['A', 'C', 'E'])
        subsetter3.subset(subset_font3)
        print(f"Glyph name subset (A,C,E): {subset_font3.getGlyphNames()}")
        
    finally:
        os.unlink(temp_file.name)

print("\n✅ Font subsetting completed - reduces file size for web use")

## Example 3: Font Format Conversion and TTX

In [ ]:
# Example 3: Font Format Conversion and TTX
from fontTools.ttLib import TTFont
from fontTools.fontBuilder import FontBuilder
from fontTools import ttx
import tempfile
import os

# Create a simple font for conversion demonstration
fb = FontBuilder(1000, isTTF=True)

# Setup glyph order
fb.setupGlyphOrder([".notdef", "A", "B"])

# Character mapping
fb.setupCharacterMap({ord('A'): 'A', ord('B'): 'B'})

# Font metadata
fb.setupNameTable({
    "familyName": "ConversionTest",
    "styleName": "Regular",
    "uniqueFontIdentifier": "ConversionTest-Regular",
    "fullName": "ConversionTest Regular",
    "psName": "ConversionTest-Regular",
    "version": "1.0"
})

# Metrics
fb.setupHorizontalMetrics({".notdef": (500, 50), "A": (600, 50), "B": (650, 50)})

# Setup required tables
fb.setupHorizontalHeader()
fb.setupOS2()
fb.setupPost()

# Note: We'll work without glyph outlines for this demonstration
# as setupGlyf requires complex glyph data structures

font = fb.font

print("=== Font Information ===")
print(f"Font tables: {sorted(font.keys())}")

# Demonstrate TTX conversion capabilities
print(f"\n=== TTX Conversion Demonstration ===")
print("FontTools TTX can convert fonts to/from XML format")
print("This allows human-readable font inspection and modification")
print("Example: ttx font.ttf creates font.ttx (XML)")
print("Example: ttx font.ttx creates font.ttf (binary)")

# Demonstrate table access and manipulation
print(f"\n=== Font Table Manipulation ===")

# Access and modify specific tables
head_table = font['head']
print(f"Original font revision: {head_table.fontRevision}")

# Modify font revision
head_table.fontRevision = 2.0
print(f"Updated font revision: {head_table.fontRevision}")

# Access name table
name_table = font['name']
print(f"Font family name: {name_table.getDebugName(1)}")

# Show glyph information
print(f"Font glyph names: {font.getGlyphNames()}")

# Demonstrate cmap table access
best_cmap = font.getBestCmap()
if best_cmap:
    print(f"Character map: {dict(best_cmap)}")

# Show font metrics
hhea_table = font['hhea']
print(f"Ascender: {hhea_table.ascent}, Descender: {hhea_table.descent}")

# OS/2 table information
os2_table = font['OS/2']
print(f"Weight class: {os2_table.usWeightClass}")
print(f"Width class: {os2_table.usWidthClass}")

# Horizontal metrics table
hmtx_table = font['hmtx']
print(f"Horizontal metrics for 'A': {hmtx_table['A']}")

# Demonstrate font serialization
print(f"\n=== Font Serialization ===")
with tempfile.NamedTemporaryFile(suffix='.ttf', delete=False) as temp_file:
    try:
        font.save(temp_file.name)
        file_size = os.path.getsize(temp_file.name)
        print(f"Font saved successfully: {file_size} bytes")
        
        # Reload to verify round-trip
        reloaded_font = TTFont(temp_file.name)
        print(f"Reloaded font has {len(reloaded_font.getGlyphNames())} glyphs")
        print(f"Glyph names: {reloaded_font.getGlyphNames()}")
        
    finally:
        os.unlink(temp_file.name)

print("\n✅ Font format conversion and table manipulation completed")

## Example 4: Font Merging and Unicode Utilities

In [ ]:
# Example 4: Font Merging and Unicode Utilities
from fontTools.merge import Merger
from fontTools.fontBuilder import FontBuilder
from fontTools.ttLib import TTFont
from fontTools.unicodedata import script, block, name
import tempfile
import os

print("=== Unicode Utilities ===")

# Demonstrate Unicode utilities
test_chars = ['A', 'α', '中', '🎵', 'ñ']
for char in test_chars:
    try:
        char_script = script(ord(char))
        char_block = block(ord(char))
        char_name = name(char, f"U+{ord(char):04X}")  # name() takes the character itself, not its code point
        print(f"'{char}' (U+{ord(char):04X}): Script={char_script}, Block={char_block}, Name={char_name}")
        assert char_name != f"U+{ord(char):04X}", f"no Unicode name for {char!r}"
    except Exception as e:
        print(f"'{char}': Error getting info - {e}")
        raise

assert (script(ord('α')), block(ord('中')), name('🎵')) == ("Grek", "CJK Unified Ideographs", "MUSICAL NOTE")

print(f"\n=== Font Merging ===")

# Create two fonts with different character sets
def create_font(name, chars, family_name):
    fb = FontBuilder(1000, isTTF=True)
    
    # Setup glyph order
    glyph_names = [".notdef"] + [f"glyph_{char}" for char in chars]
    fb.setupGlyphOrder(glyph_names)
    
    # Character mapping
    char_map = {ord(char): f"glyph_{char}" for char in chars}
    fb.setupCharacterMap(char_map)
    
    # Font metadata
    fb.setupNameTable({
        "familyName": family_name,
        "styleName": "Regular",
        "uniqueFontIdentifier": f"{family_name}-Regular",
        "fullName": f"{family_name} Regular"
    })
    
    # Metrics
    metrics = {".notdef": (500, 50)}
    for char in chars:
        metrics[f"glyph_{char}"] = (600, 50)
    fb.setupHorizontalMetrics(metrics)
    
    # Setup required tables
    fb.setupHorizontalHeader()
    fb.setupOS2()
    fb.setupPost()
    
    return fb.font

# Create Font 1: Latin characters
font1 = create_font("font1", ['A', 'B', 'C'], "LatinFont")
print(f"Font 1 glyphs: {font1.getGlyphNames()}")
print(f"Font 1 character map: {dict(font1.getBestCmap())}")

# Create Font 2: Different characters
font2 = create_font("font2", ['X', 'Y', 'Z'], "AlphaFont")
print(f"Font 2 glyphs: {font2.getGlyphNames()}")
print(f"Font 2 character map: {dict(font2.getBestCmap())}")

# Merge fonts using Merger
merger = Merger()

# Configure merger options
merger.duplicateGlyphsPerFont = False  # Don't duplicate glyphs

print(f"\n=== Merging Process ===")

try:
    # Save fonts to temporary files for merging
    with tempfile.NamedTemporaryFile(suffix='.ttf', delete=False) as f1:
        with tempfile.NamedTemporaryFile(suffix='.ttf', delete=False) as f2:
            font1.save(f1.name)
            font2.save(f2.name)
            
            # Load fonts for merging
            ttf1 = TTFont(f1.name)
            ttf2 = TTFont(f2.name)
            
            print(f"Before merge - Font 1: {len(ttf1.getGlyphNames())} glyphs")
            print(f"Before merge - Font 2: {len(ttf2.getGlyphNames())} glyphs")
            
            # Perform merge (Merger.merge takes font FILE paths)
            merged_font = merger.merge([f1.name, f2.name])
            
            print(f"After merge: {len(merged_font.getGlyphNames())} glyphs")
            print(f"Merged glyph names: {merged_font.getGlyphNames()}")
            print(f"Merged character map: {dict(merged_font.getBestCmap())}")
            
            # Save merged font
            with tempfile.NamedTemporaryFile(suffix='.ttf', delete=False) as merged_file:
                merged_font.save(merged_file.name)
                print(f"Merged font saved: {os.path.getsize(merged_file.name)} bytes")
                
                # Verify merged font
                verification = TTFont(merged_file.name)
                cmap = verification.getBestCmap()
                available_chars = [chr(code) for code in cmap.keys()]
                print(f"Available characters in merged font: {available_chars}")
                assert sorted(available_chars) == ['A', 'B', 'C', 'X', 'Y', 'Z'], available_chars
                
                os.unlink(merged_file.name)
            
            os.unlink(f1.name)
            os.unlink(f2.name)
            
except Exception as e:
    print(f"Merge error: {e}")
    print("Note: Font merging requires compatible font structures")
    raise

# Demonstrate character set analysis
print(f"\n=== Character Set Analysis ===")

# Analyze what scripts are covered
sample_text = "Hello World! 123 αβγ"
covered_scripts = set()
for char in sample_text:
    if char.strip():
        try:
            char_script = script(ord(char))
            covered_scripts.add(char_script)
        except:
            pass
            raise

print(f"Text: '{sample_text}'")
print(f"Scripts covered: {sorted(covered_scripts)}")

print("\n✅ Font merging and Unicode utilities completed")

## Example 5: Advanced Font Features and OpenType

In [ ]:
# Example 5: Advanced Font Features and OpenType
from fontTools.fontBuilder import FontBuilder
from fontTools.ttLib import TTFont
from fontTools.misc.transform import Transform
import tempfile
import os
import math

print("=== Advanced Font Building ===")

# Create a more sophisticated font with advanced features
fb = FontBuilder(1000, isTTF=True)

# Define comprehensive glyph set
glyph_list = [
    ".notdef", "space", "A", "B", "C", "D", "E", 
    "a", "b", "c", "d", "e", "f", "i", "l",
    "period", "comma", "hyphen", "zero", "one", "two"
]

# Setup glyph order
fb.setupGlyphOrder(glyph_list)

# Create character map
char_map = {
    ord(' '): 'space', ord('A'): 'A', ord('B'): 'B', ord('C'): 'C', 
    ord('D'): 'D', ord('E'): 'E', ord('a'): 'a', ord('b'): 'b', 
    ord('c'): 'c', ord('d'): 'd', ord('e'): 'e', ord('f'): 'f',
    ord('i'): 'i', ord('l'): 'l', ord('.'): 'period', ord(','): 'comma',
    ord('-'): 'hyphen', ord('0'): 'zero', ord('1'): 'one', ord('2'): 'two'
}
fb.setupCharacterMap(char_map)

# Comprehensive font metadata
fb.setupNameTable({
    "familyName": "AdvancedTest",
    "styleName": "Regular", 
    "uniqueFontIdentifier": "AdvancedTest-Regular-1.0",
    "fullName": "AdvancedTest Regular",
    "psName": "AdvancedTest-Regular",
    "version": "1.000"
})

# Advanced horizontal metrics with varied widths
advanced_metrics = {
    ".notdef": (500, 50), "space": (250, 0), "A": (700, 50), "B": (650, 50),
    "C": (680, 50), "D": (720, 50), "E": (580, 50), "a": (560, 50),
    "b": (580, 50), "c": (520, 50), "d": (580, 50), "e": (540, 50),
    "f": (320, 20), "i": (260, 50), "l": (260, 50), "period": (260, 50),
    "comma": (260, 50), "hyphen": (340, 30), "zero": (560, 50),
    "one": (560, 50), "two": (560, 50)
}
fb.setupHorizontalMetrics(advanced_metrics)

# Setup required tables
fb.setupHorizontalHeader()
fb.setupOS2()
fb.setupPost()

# Create the font
advanced_font = fb.font

print(f"Created font with {len(advanced_font.getGlyphNames())} glyphs")
print(f"Font tables: {sorted(advanced_font.keys())}")

# Advanced table inspection
print(f"\n=== Font Table Details ===")

# Head table details
head = advanced_font['head']
print(f"Head table - Units per EM: {head.unitsPerEm}")
print(f"Head table - Font revision: {head.fontRevision}")

# OS/2 table details
os2 = advanced_font['OS/2']
print(f"OS/2 - Weight class: {os2.usWeightClass}")
print(f"OS/2 - Width class: {os2.usWidthClass}")

# Name table details
name_table = advanced_font['name']
print(f"Font family: {name_table.getDebugName(1)}")
print(f"Font style: {name_table.getDebugName(2)}")

# Character map analysis
best_cmap = advanced_font.getBestCmap()
print(f"Character map entries: {len(best_cmap)}")
print(f"Sample mappings: {dict(list(best_cmap.items())[:5])}")

# Horizontal metrics analysis
hmtx = advanced_font['hmtx']
print(f"Metrics - A: width={hmtx['A'][0]}")
print(f"Metrics - space: width={hmtx['space'][0]}")

# Font validation
print(f"\n=== Font Validation ===")

# Check available tables
available_tables = list(advanced_font.keys())
core_tables = ['cmap', 'head', 'hhea', 'hmtx', 'maxp', 'name', 'OS/2', 'post']
present_core = [table for table in core_tables if table in available_tables]
print(f"Core font tables present: {len(present_core)}/{len(core_tables)}")
print(f"Present: {present_core}")

# Advanced FontTools capabilities
print(f"\n=== Advanced FontTools Capabilities ===")

# Transform demonstration for coordinate operations
identity = Transform()
scale2x = Transform(2.0, 0, 0, 2.0, 0, 0)  # Scale by 2x
rotate45 = Transform().rotate(math.radians(45))  # 45 degree rotation
translate = Transform().translate(100, 50)  # Translation
print(f"Identity transform: {identity}")
print(f"2x scale transform: {scale2x}")
print(f"45° rotation transform: {rotate45}")
print(f"Translation transform: {translate}")

# Glyph categorization
glyph_names = advanced_font.getGlyphNames()
uppercase = [g for g in glyph_names if g.isupper()]
lowercase = [g for g in glyph_names if g.islower()]
digits = [g for g in glyph_names if g.isdigit()]
special = [g for g in glyph_names if not (g.isupper() or g.islower() or g.isdigit())]
print(f"Uppercase glyphs: {uppercase}")
print(f"Lowercase glyphs: {lowercase}")
print(f"Digit glyphs: {digits}")
print(f"Special glyphs: {special}")

# Font metrics and layout information
hhea = advanced_font['hhea']
print(f"Layout metrics - Ascender: {hhea.ascent}, Descender: {hhea.descent}")
print(f"Line gap: {hhea.lineGap}")

# Character encoding analysis
print(f"\n=== Character Encoding Analysis ===")
cmap_entries = list(best_cmap.items())
print(f"Total mapped characters: {len(cmap_entries)}")
unicode_ranges = {}
for codepoint, glyph in cmap_entries:
    if 0x0020 <= codepoint <= 0x007E:  # Printable ASCII
        unicode_ranges.setdefault('ASCII', []).append(chr(codepoint))
    elif 0x0030 <= codepoint <= 0x0039:  # Digits
        unicode_ranges.setdefault('Digits', []).append(chr(codepoint))

for range_name, chars in unicode_ranges.items():
    print(f"{range_name} characters: {''.join(chars)}")

print("\n✅ Advanced font features and OpenType analysis completed")